# 02 — RAG

**AI Engineer — hands-on session**

In notebook 01 we ended with a problem:

> Context is finite, and you pay for it by the token. We cannot paste a company's entire documentation into every question.

**RAG** — Retrieval Augmented Generation — is the answer. And it is less magical than the name suggests:

> Find the handful of paragraphs most likely to answer the question, paste **those** into the prompt, and ask normally.

That is it. The whole field is about doing the *finding* well.

---

### Before you start

Same `GROQ_API_KEY` secret as notebook 01. The install below pulls a few hundred megabytes, so run it before the session starts.

In [ ]:
%pip install -q --upgrade openai sentence-transformers chromadb plotly scikit-learn "gradio>=5"

In [ ]:
import getpass
import importlib
import os
from pathlib import Path

# Outside the git repo (ie. in Colab) fetch the documents and the plotting helpers.
if not os.path.exists(".git"):
    !git clone -q --depth 1 https://github.com/ingmiguelfernando/AIExperiment.git /tmp/aiexperiment
    !cp -r /tmp/aiexperiment/knowledge-base /tmp/aiexperiment/helpers.py .

import helpers

importlib.reload(helpers)

In [ ]:
from openai import OpenAI

BASE_URL = "https://api.groq.com/openai/v1"
MODEL = "openai/gpt-oss-20b"


def get_secret(name):
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass

    return os.environ.get(name) or getpass.getpass(f"{name}: ")


client = OpenAI(api_key=get_secret("GROQ_API_KEY"), base_url=BASE_URL)

print("Model:", MODEL)

---
## 1. The problem, demonstrated

**KiwiAir** is the fictional airline from notebook 01. We now have its internal documents: company background, employee records, product specs and customer contracts.

Let's ask the model something that is answered in those documents, without giving it the documents.

In [ ]:
QUESTION = "Who is Wiremu Katene, and what unusual authority does he hold at KiwiAir?"

without_rag = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": QUESTION}],
)

print(without_rag.choices[0].message.content)

> Nothing. And that is the **correct** outcome — KiwiAir does not exist, so no amount of training data could have taught the model this.
>
> This is exactly why we use fictional data for the demo: any correct answer later can only have come from the documents we supplied. There is no other source.

---
## 2. The knowledge base

Sixteen Markdown files in four folders. Nothing clever — just text on disk.

In [ ]:
documents = [
    {
        "doc_type": path.parent.name,
        "name": path.stem,
        "text": path.read_text(encoding="utf-8"),
    }
    for path in sorted(Path("knowledge-base").glob("*/*.md"))
]

for document in documents:
    print(f"{document['doc_type']:<10} {document['name'][:45]:<48} {len(document['text']):>6} chars")

print(f"\n{len(documents)} documents")

---
## 3. Chunking

We do not store whole documents. A 3,000-word contract would swamp the prompt and most of it would be irrelevant to any single question.

So we cut everything into small overlapping pieces. The **overlap** matters: without it, a sentence that straddles a boundary gets cut in half and neither piece makes sense.

In [ ]:
CHUNK_SIZE = 700
OVERLAP = 120

chunks = []
for document in documents:
    text = document["text"]
    for start in range(0, len(text), CHUNK_SIZE - OVERLAP):
        piece = text[start : start + CHUNK_SIZE].strip()
        if piece:
            chunks.append({**document, "text": piece})

print(f"{len(documents)} documents -> {len(chunks)} chunks\n")
print(chunks[20]["doc_type"], "/", chunks[20]["name"])
print("-" * 70)
print(chunks[20]["text"])

> Real systems split on structure — headings, paragraphs, sentences — rather than a fixed character count. We are using the blunt version so the idea stays visible.

---
## 4. Embeddings: turning meaning into numbers

This is the one genuinely new idea in RAG.

An **embedding model** reads a piece of text and returns a list of numbers — a point in space. It is trained so that texts *meaning* similar things land near each other, even when they share no words at all.

We use a small open model that runs right here, on this machine. No API, no key, no data leaving the runtime.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")

vector = embedder.encode("How much does a flexible ticket cost?")

print("Dimensions:", len(vector))
print("First 8   :", vector[:8].round(3))

### Why those numbers are useful

Three sentences. The first two mean nearly the same thing but share almost no vocabulary. The third is unrelated.

Keyword search would rank the first two as *unrelated*. Watch what the embeddings say:

In [ ]:
sentences = [
    "How much does a flexible ticket cost?",
    "What is the price of a changeable fare?",
    "The maintenance facility is in Palmerston North.",
]

vectors = embedder.encode(sentences, normalize_embeddings=True)
similarity = vectors @ vectors.T  # cosine similarity, because the vectors are normalised

for i, row in enumerate(similarity):
    print(f"{sentences[i][:45]:<48}", " ".join(f"{value:5.2f}" for value in row))

> ### 💡 The idea worth remembering
> **Meaning became geometry.** "Flexible ticket" and "changeable fare" are close together despite sharing no words; the maintenance sentence sits far from both.
>
> Once meaning is a position in space, *"find me the most relevant paragraph"* becomes *"find me the nearest point"* — and computers are extremely good at that.

---
## 5. The vector store

We embed every chunk and keep the results somewhere we can search by proximity. That is all a "vector database" is.

Chroma runs inside this process — no server, no cloud account.

In [ ]:
import chromadb

texts = [chunk["text"] for chunk in chunks]
embeddings = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)

collection = chromadb.Client().get_or_create_collection("kiwiair")
collection.upsert(
    ids=[str(i) for i in range(len(chunks))],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{"doc_type": c["doc_type"], "name": c["name"]} for c in chunks],
)

print(f"\n{collection.count()} chunks stored, {embeddings.shape[1]} dimensions each")

---
## 6. Let's actually look at it

Those vectors have 384 dimensions, which nobody can picture. **t-SNE** squashes them down to 2, keeping near things near.

Nothing below knows which folder a chunk came from — the colours are added afterwards, purely so we can check. If the embedding model understood the content, the colours should separate on their own.

In [ ]:
labels = [chunk["doc_type"] for chunk in chunks]
hover = [f"<b>{c['name']}</b><br>{c['text'][:120]}..." for c in chunks]

helpers.plot_vectors(embeddings, labels, hover, dimensions=2)

Hover over the points to read the chunks. Now the same thing in 3D — drag to rotate:

In [ ]:
helpers.plot_vectors(embeddings, labels, hover, dimensions=3, title="Same vectors, one more dimension")

> The clusters are the whole point. **We never told it what a contract is.** It read the text and the contracts ended up together, the employee records ended up together, and so on.
>
> Watch the chunks that sit between clusters — those are usually documents that genuinely span two topics, like a contract that names the employee who owns it.

---
## 7. Retrieval

Now the actual search. Embed the question with the **same model**, then ask for the nearest chunks.

In [ ]:
def retrieve(question, how_many=5):
    query_vector = embedder.encode([question], normalize_embeddings=True)
    found = collection.query(query_embeddings=query_vector.tolist(), n_results=how_many)

    return list(zip(found["documents"][0], found["metadatas"][0], found["distances"][0]))


for text, meta, distance in retrieve(QUESTION):
    print(f"[{distance:.3f}] {meta['doc_type']} / {meta['name']}")
    print(f"        {text[:110].replace(chr(10), ' ')}...\n")

> The distance is how far each chunk sits from the question in that 384-dimensional space. Smaller is closer.
>
> **No language model has been involved yet.** This is pure geometry.

---
## 8. Putting it in the prompt

Here is the part that surprises people. After all that machinery, what we do with the results is... paste them into the message list.

In [ ]:
def build_messages(question):
    context = "\n\n---\n\n".join(text for text, _, _ in retrieve(question))

    return [
        {
            "role": "system",
            "content": (
                "You answer questions about KiwiAir using only the context provided. "
                "If the context does not contain the answer, say so. Be brief."
            ),
        },
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]


messages = build_messages(QUESTION)

print(messages[1]["content"][:900], "\n\n[...]")

> ### 💡 Remember the "what is my name?" experiment
> We fixed the model's missing memory by **putting the information into the message list**. RAG is that same move.
>
> The only new thing is *how we choose what to put in*. Everything else — embeddings, vector stores, t-SNE — exists to answer one question: **which paragraphs are worth the tokens?**

In [ ]:
with_rag = client.chat.completions.create(model=MODEL, messages=messages)

print("WITHOUT RAG\n" + "=" * 70)
print(without_rag.choices[0].message.content)

print("\n\nWITH RAG\n" + "=" * 70)
print(with_rag.choices[0].message.content)

### It also has to know when to refuse

A retrieval system always returns *something* — the nearest chunks exist even when none of them are relevant. Grounding the model properly means it should decline rather than invent.

In [ ]:
off_topic = "What is KiwiAir's policy on pet rabbits in the cabin?"

answer = client.chat.completions.create(model=MODEL, messages=build_messages(off_topic))

print(answer.choices[0].message.content)

---
## 9. The whole thing as a chat

Same Gradio loop as notebook 01. The only difference is one line: we retrieve before we ask.

Try: *What does KiwiFlex cost?* · *What happens if the cold chain is broken?* · *Who negotiated the Series B?*

In [ ]:
def chat(message, history):
    stream = client.chat.completions.create(
        model=MODEL,
        messages=build_messages(message),
        stream=True,
    )

    partial = ""
    for chunk in stream:
        partial += chunk.choices[0].delta.content or ""
        yield partial


helpers.launch_chat(chat, title="Ask the KiwiAir documents")

---
## Recap

| Step | What happens | Cost |
|---|---|---|
| Chunk | Cut documents into overlapping pieces | Once |
| Embed | Turn each piece into a point in space | Once |
| Store | Keep the points somewhere searchable | Once |
| Retrieve | Find the nearest points to the question | Every question |
| Generate | Paste those pieces into the prompt and ask | Every question |

The first three happen when documents change. The last two happen per question, and only the last one costs tokens.

### What this buys you

- The model can answer about documents it never saw in training
- You can point at the exact source of an answer
- Updating knowledge means re-indexing a file, not retraining a model
- You pay for five paragraphs instead of an entire document set

### Where it breaks

- If retrieval misses, the model answers from nothing — quality is capped by the search, not the model
- Questions spanning many documents (*"summarise every contract"*) do not fit the pattern
- Chunk boundaries can cut an answer in half

> **The punchline:** every trick in this session — chat history, tools, RAG — is the same move. Decide what text goes in the prompt.